# 07 · Modelagem bônus: comparação de algoritmos de agrupamento

**Complementar à CRISP-DM fase 4, modelagem** · Turismo e Hotelaria

---

!!! info "Como este notebook se encaixa no projeto"
    A clusterização oficial do projeto (K-Means, k=5) está em
    `04-0-modelagem-cluster.ipynb` e `04-1-modelagem-cotovelo.ipynb`, e é ela que
    alimenta a avaliação (`05-avaliacao.ipynb`) e a implantação
    (`06-implantacao.ipynb`). Este notebook é um experimento complementar: compara
    outros algoritmos de agrupamento (K-Means, Mini Batch K-Means, Bisecting K-Means
    e Birch) sobre uma amostra controlada da base, para checar se outro algoritmo ou
    outro número de grupos mudaria a conclusão. O resultado é guardado em
    `models/model_bonus_hotel_<algoritmo>.joblib`, separado do modelo oficial, e
    não substitui nenhuma etapa das fases 4 a 6. A seção final "Como ler o que
    saiu" discute o que esse experimento encontrou e por que a equipe manteve a
    decisão oficial mesmo assim.


In [ ]:
import sys
import time
import warnings
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown
from sklearn.cluster import Birch, BisectingKMeans, KMeans, MiniBatchKMeans
from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

warnings.filterwarnings("ignore")

# O notebook roda a partir de `notebooks/`; a raiz entra no path para que
# `import src...` funcione sem instalar o pacote.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

(RAIZ / "docs" / "imagens").mkdir(parents=True, exist_ok=True)

In [ ]:
data_path = Path("../data/processed/hotel_bookings_tratado.csv")
preprocessed_path = Path("../data/processed/hotel_bookings_preprocessed.csv")
dict_path = Path("../data/processed/dicionario.csv")

In [ ]:
df = pd.read_csv(data_path, sep=";")
df_preprocessed = pd.read_csv(preprocessed_path, sep=";")

display(
    Markdown(
        f"### Conjunto de dados, " f"{len(df):,} reservas × {df.shape[1]} colunas".replace(",", ".")
    )
)
display(df.head())

df_dict = pd.read_csv(dict_path, sep=";").assign(coluna=lambda tabela: tabela["variavel"])

display(Markdown("### Dicionário de dados"))
display(df_dict.head())

display(
    Markdown(
        "Tipos declarados: "
        + " · ".join(
            f"**{tipo}** {quantidade}"
            for tipo, quantidade in df_dict["Tipo"].value_counts().items()
        )
    )
)

In [ ]:
colunas_derivadas = {
    "arrival_date",
    "total_nights",
    "total_guests",
    "room_changed",
    "has_special_requests",
}

colunas_base = set(df.columns)
colunas_dicionario = set(df_dict["coluna"].dropna())

fora_do_dicionario = sorted(colunas_base - colunas_dicionario - colunas_derivadas)

display(
    Markdown(
        f"- colunas derivadas aceitas: **{len(colunas_base & colunas_derivadas)}**\n"
        f"- colunas da base sem descrição: **{len(fora_do_dicionario)}** "
        f"{fora_do_dicionario}\n"
        f"- matriz pré-processada: **{df_preprocessed.shape[0]:,} x "
        f"{df_preprocessed.shape[1]}**".replace(",", ".")
    )
)

assert not fora_do_dicionario

In [ ]:
# Selecionando os atributos da base hoteleira

useless_columns = [
    "is_canceled",
    "reservation_status",
    "reservation_status_date",
    "arrival_date",
    "agent",
    "company",
]


def colunas_do_tipo(*tipos):
    return [
        coluna
        for coluna in df_dict.query("Tipo in @tipos")["coluna"].dropna()
        if coluna in df.columns and coluna not in useless_columns
    ]


continuous_columns = colunas_do_tipo(
    "Contínua",
    "Discreta",
    "Binária",
)

ordinal_columns = colunas_do_tipo("Ordinal")
nominal_columns = colunas_do_tipo("Nominal")

derived_binary = [
    "room_changed",
    "has_special_requests",
]

continuous_columns = list(dict.fromkeys(continuous_columns + derived_binary))

display(
    Markdown(
        f"- contínuas/discretas/binárias: **{len(continuous_columns)}**\n"
        f"- ordinais: **{len(ordinal_columns)}**\n"
        f"- nominais: **{len(nominal_columns)}**\n"
        f"- fora da modelagem: **{len(useless_columns)}**"
    )
)

In [ ]:
# A matriz usada pelos algoritmos é a mesma matriz pré-processada do 03,
# já no intervalo de 0 a 1 e com as nominais em one-hot.
X_ref = df_preprocessed.copy()

print(
    f"Matriz pré-processada: {X_ref.shape[0]:,} reservas x " f"{X_ref.shape[1]} atributos".replace(
        ",", "."
    )
)

In [ ]:
models = [
    (
        "k-means",
        KMeans(
            n_init=5,
            random_state=42,
        ),
        {
            "n_clusters": [2, 3, 4, 5],
            "init": ["k-means++", "random"],
        },
    ),
    (
        "mini batch k-means",
        MiniBatchKMeans(
            n_init=5,
            batch_size=1024,
            random_state=42,
        ),
        {
            "n_clusters": [2, 3, 4, 5],
        },
    ),
    (
        "bisecting k-means",
        BisectingKMeans(
            random_state=42,
        ),
        {
            "n_clusters": [2, 3, 4, 5],
            "bisecting_strategy": [
                "biggest_inertia",
                "largest_cluster",
            ],
        },
    ),
    (
        "birch",
        Birch(),
        {
            "n_clusters": [2, 3, 4, 5],
            "threshold": [0.1, 0.3, 0.5],
        },
    ),
]

In [ ]:
# Avaliação dos modelos em amostra controlada

TAMANHO_AMOSTRA = min(5_000, len(X_ref))
rng = np.random.default_rng(42)
indices = rng.choice(
    len(X_ref),
    size=TAMANHO_AMOSTRA,
    replace=False,
)

X_modelo = X_ref.iloc[indices].to_numpy(dtype=float)

resultados = []

for model_name, model_obj, model_params in models:
    for params in ParameterGrid(model_params):
        inicio = time.time()

        modelo = model_obj.set_params(**params)
        modelo.fit(X_modelo)
        labels = modelo.labels_

        silhueta = silhouette_score(
            X_modelo,
            labels,
            metric="euclidean",
        )

        resultados.append(
            {
                "modelo": model_name,
                "parametros": params,
                "silhueta": silhueta,
                "grupos": len(np.unique(labels)),
                "tempo": time.time() - inicio,
            }
        )

resultados = pd.DataFrame(resultados).sort_values(
    "silhueta",
    ascending=False,
)

display(resultados)

In [ ]:
# Selecionando o melhor modelo

melhor = resultados.iloc[0]

winner = melhor["modelo"]
best_params = melhor["parametros"]

print(f"Melhor modelo: {winner}")
print(f"Parâmetros: {best_params}")
print(f"Silhueta: {melhor['silhueta']:.3f}")
print(f"Grupos: {int(melhor['grupos'])}")

In [ ]:
# Treinando o melhor modelo na matriz completa

model_name, model, _ = [item for item in models if item[0] == winner][0]

modelo_final = model.set_params(**best_params)
modelo_final.fit(X_ref.to_numpy(dtype=float))

labels = modelo_final.labels_

X_t = X_ref.to_numpy(dtype=float)

display(
    Markdown(
        f"Modelo final: **{model_name}**  \n"
        f"Silhueta: **{silhouette_score(X_t, labels):.3f}**  \n"
        f"Grupos formados: **{len(np.unique(labels))}**"
    )
)

In [ ]:
# Perfil dos segmentos de reservas hoteleiras

perfil = (
    df.assign(segmento=labels)
    .groupby("segmento")
    .agg(
        reservas=("hotel", "size"),
        taxa_cancelamento=("is_canceled", "mean"),
        lead_time_mediano=("lead_time", "median"),
        noites_mediana=("total_nights", "median"),
        hospedes_mediana=("total_guests", "median"),
        adr_mediana=("adr", "median"),
        solicitacoes_mediana=("total_of_special_requests", "median"),
    )
    .assign(participacao=lambda tabela: tabela["reservas"] / len(df))
    .sort_values("taxa_cancelamento", ascending=False)
)

display(Markdown("### Perfil dos segmentos"))
display(
    perfil.style.format(
        {
            "taxa_cancelamento": "{:.1%}",
            "participacao": "{:.1%}",
            "lead_time_mediano": "{:.0f}",
            "noites_mediana": "{:.0f}",
            "hospedes_mediana": "{:.0f}",
            "adr_mediana": "{:.2f}",
            "solicitacoes_mediana": "{:.0f}",
        }
    )
)

In [ ]:
# Salvando o modelo bonus de hotelaria

PASTA_MODELOS = RAIZ / "models"
PASTA_PROCESSADOS = RAIZ / "data" / "processed"

PASTA_MODELOS.mkdir(parents=True, exist_ok=True)
PASTA_PROCESSADOS.mkdir(parents=True, exist_ok=True)

nome_modelo = winner.replace(" ", "_").replace("-", "_")

modelo_path = PASTA_MODELOS / f"model_bonus_hotel_{nome_modelo}.joblib"

joblib.dump(modelo_final, modelo_path)

pd.DataFrame(
    {
        "segmento": labels,
    }
).to_csv(
    PASTA_PROCESSADOS / "segmentos_bonus.csv",
    sep=";",
    index=False,
)

perfil.to_csv(
    PASTA_PROCESSADOS / "perfil_segmentos_bonus.csv",
    sep=";",
    encoding="utf-8",
)

display(
    Markdown(
        f"Modelo salvo em `{modelo_path}`  \n"
        "Rótulos salvos em `data/processed/segmentos_bonus.csv`  \n"
        "Perfil salvo em `data/processed/perfil_segmentos_bonus.csv`"
    )
)

## Visualização das reservas segmentadas

Os 98 atributos não dão para desenhar num gráfico. Para ver os grupos
encontrados por este notebook, projetamos as reservas nos dois primeiros
componentes de PCA e de UMAP, já calculados em
`04-2-modelagem-reducao.ipynb`, e coloramos cada ponto pelo segmento deste
notebook (não pelo segmento oficial do `04-0`).


In [ ]:
reservas_segmentadas = df.assign(segmento=labels)

pca = pd.read_csv(RAIZ / "data" / "processed" / "hotel_bookings_pca.csv", sep=";")
umap_df = pd.read_csv(RAIZ / "data" / "processed" / "hotel_bookings_umap.csv", sep=";")

fig, eixos = plt.subplots(1, 2, figsize=(13, 5))
for ax, base, eixo_x, eixo_y, titulo in [
    (eixos[0], pca, pca.columns[0], pca.columns[1], "Dois primeiros componentes do PCA"),
    (
        eixos[1],
        umap_df,
        umap_df.columns[0],
        umap_df.columns[1],
        "Dois primeiros componentes do UMAP",
    ),
]:
    dispersao = ax.scatter(
        base[eixo_x],
        base[eixo_y],
        c=labels,
        cmap="tab10",
        s=6,
        alpha=0.5,
    )
    ax.set(title=titulo, xlabel=eixo_x, ylabel=eixo_y)
fig.colorbar(dispersao, ax=eixos, label="segmento (07_modelagem_bonus)", shrink=0.8)
fig.savefig(
    RAIZ / "docs" / "imagens" / "modelagem-bonus-segmentos.png",
    dpi=150,
    bbox_inches="tight",
)
plt.show()

reservas_segmentadas.head()

## Como ler o que saiu

!!! warning "Reconciliação, não correção"
    Esta seção compara o resultado deste experimento bônus com a decisão
    oficial do `04-0-modelagem-cluster.ipynb` (K-Means, k=5). Não é uma
    correção da modelagem oficial: é um registro de que a equipe viu essa
    diferença e decidiu, de forma consciente, manter k=5.

* **Por silhueta pura, k=2 ganha de k=5, mas por uma margem mínima.** Na
  amostra de 5.000 reservas, o melhor resultado geral é o Mini Batch
  K-Means com `n_clusters=2` (silhueta **0,1249**); o melhor resultado com
  `n_clusters=5` é o K-Means com `n_clusters=5` (silhueta **0,1243**). A
  diferença é de **0,0007** de silhueta, bem dentro do ruído esperado entre
  uma amostra de 5.000 reservas e as 119.390 da base inteira. Treinado na
  base inteira, o Mini Batch K-Means com `n_clusters=2` fica com silhueta
  **0,100**, abaixo da silhueta de **0,1257** do K-Means k=5 oficial
  (ver [05-avaliacao.ipynb](05-avaliacao.ipynb) e
  [docs/avaliacao.md](../docs/avaliacao.md)): a vantagem de k=2
  nem se confirma na base completa.

* **Mesmo se k=2 vencesse por uma margem maior, a equipe manteria k=5.** O
  [Canvas do Problema](../docs/pbl/canvas-do-problema.md) (campos 4, 6 e 10)
  fixa a regra de negócio deste projeto: a empresa financia **uma ação
  diferenciada por grupo, no máximo cinco**. Um resultado com k=2 dá duas
  ações possíveis; um resultado com k=5 dá cinco. A silhueta mede separação
  geométrica entre grupos, não quantas ações de negócio diferentes a
  diretoria consegue tomar. Entre um modelo com separação ligeiramente
  melhor e dois grupos, e um modelo com separação praticamente igual, mas
  cinco grupos (cada um com perfil e ação próprios, ver
  [docs/criterios-sucesso.md](../docs/criterios-sucesso.md#acao-por-segmento)),
  a segunda opção atende melhor ao objetivo de negócio do Canvas.

* **A silhueta baixa nos dois casos (0,10 a 0,12) é esperada, não é um
  problema deste experimento.** [docs/avaliacao.md](../docs/avaliacao.md) já
  registra a mesma causa para o modelo oficial: o espaço de 98 colunas, com
  muito One-Hot, dilui a separação geométrica entre grupos. Trocar o
  algoritmo de agrupamento não resolve isso, porque o problema está no
  espaço de atributos, não no algoritmo.

* **Conclusão.** Este experimento bônus confirma, por um caminho
  independente, que a escolha de k=5 no `04-0` não estava deixando uma
  separação geométrica muito melhor na mesa: a melhor alternativa (k=2) é
  praticamente equivalente em silhueta e entrega menos granularidade de
  negócio. A decisão oficial de manter k=5 continua de pé.
